# Prompt Management & Observability

Day 4 of Week 1. A prompt that works in development may silently degrade in production. Tokens cost money — spending $200/day on a pipeline you cannot observe is an engineering liability. This notebook instruments every LLM call with Langfuse: full request/response traces, cost and latency per call, prompt version management with rollback, and a dashboard that makes the pipeline's behaviour visible to engineers and product teams alike. Observability is not optional at principal level — it is the difference between a system you can debug and one you can only reboot.

Setup:

In [ ]:
#| echo: false
import os, json, time
import numpy as np
from dotenv import load_dotenv
load_dotenv()

import openai
from pydantic import BaseModel
from typing import Optional, Type

PRICES = {
    "gpt-4o":      {"input": 2.50,  "output": 10.00},
    "gpt-4o-mini": {"input": 0.15,  "output": 0.60},
}

class LLMClient:
    def __init__(self, model="gpt-4o-mini", temperature=0.0):
        self.model = model; self.temperature = temperature
        self._client = openai.OpenAI()
        self._in = 0; self._out = 0

    def complete(self, messages, *, response_format=None):
        if response_format is not None:
            resp = self._client.beta.chat.completions.parse(
                model=self.model, messages=messages,
                temperature=self.temperature, response_format=response_format)
        else:
            resp = self._client.chat.completions.create(
                model=self.model, messages=messages, temperature=self.temperature)
        if resp.usage:
            self._in += resp.usage.prompt_tokens; self._out += resp.usage.completion_tokens
        if response_format is not None: return resp.choices[0].message.parsed
        return resp.choices[0].message.content

    @property
    def total_cost(self):
        if self.model not in PRICES: return 0.0
        p = PRICES[self.model]
        return (self._in * p["input"] + self._out * p["output"]) / 1_000_000

llm = LLMClient()

## What Observability Means for LLM Systems

Traditional software observability rests on three pillars: logs, metrics, and traces. LLM systems need all three, plus two additional dimensions that standard APM tools do not cover.

**Traces** record the full call graph for a single request: which components ran, with what inputs and outputs, how long each took, and what they cost. For a RAG pipeline, a trace shows: (a) retrieval — which chunks were returned from ChromaDB, (b) prompt construction — the exact text sent to the model, (c) generation — the model's response. Without a trace, debugging a wrong answer requires guessing which of these three stages failed.

<br>

**Metrics** aggregate over time: p50/p95/p99 latency, cost per query, error rate, cache hit rate. For LLM workloads, two metrics matter above all others: (1) cost per query — if this spikes, a prompt change is sending more tokens than expected; (2) p95 generation latency — if this degrades, the model is receiving longer context than expected or the API is under load.

<br>

**Prompt versions** are the LLM-specific dimension with no traditional equivalent. Because the prompt is effectively the program, you need to record which exact prompt version was active for every request. When a customer reports a bad answer on Tuesday, you need to know: was that the v3 system prompt or the v4 one we deployed on Monday? Without prompt versioning, root cause analysis is impossible.

<br>

**Stochastic outputs** mean you cannot spot-check quality by reading logs — you need aggregate quality metrics attached to traces. This is where [notebook 04](/courses/llm-eng/04-eval-concepts.html) connects: we attach RAGAS faithfulness scores and LLM judge scores as metadata on each trace, making quality visible in the same dashboard as cost and latency.

## Langfuse Setup

Langfuse is an open-source LLM observability platform. We run it locally with Docker Compose before connecting from Python.

**Docker Compose setup** (save as `docker-compose.yml` in your project root, then run `docker compose up -d`):

```yaml
version: "3"
services:
  langfuse:
    image: langfuse/langfuse:2
    ports: ["3000:3000"]
    environment:
      DATABASE_URL: postgresql://postgres:postgres@db/langfuse
      NEXTAUTH_SECRET: changeme
      SALT: changeme
      NEXTAUTH_URL: http://localhost:3000
    depends_on: [db]
  db:
    image: postgres:15
    environment:
      POSTGRES_PASSWORD: postgres
      POSTGRES_DB: langfuse
    volumes:
      - langfuse-db:/var/lib/postgresql/data
volumes:
  langfuse-db:
```

Once running, open `http://localhost:3000`, create an account, and generate API keys under **Settings → API Keys**. Add them to your `.env`:

```bash
LANGFUSE_PUBLIC_KEY=pk-lf-...
LANGFUSE_SECRET_KEY=sk-lf-...
LANGFUSE_HOST=http://localhost:3000
```

Then install the Python client:

```bash
uv add langfuse
```

Initializing the Langfuse client:

In [ ]:
from langfuse import Langfuse

langfuse = Langfuse(
    public_key=os.environ.get("LANGFUSE_PUBLIC_KEY", "pk-lf-placeholder"),
    secret_key=os.environ.get("LANGFUSE_SECRET_KEY", "sk-lf-placeholder"),
    host=os.environ.get("LANGFUSE_HOST", "http://localhost:3000"),
)
print("Langfuse client initialized.")

## Instrumenting LLMClient with Langfuse

We extend `LLMClient` to an `ObservableLLMClient` subclass. Every `complete` call is wrapped in a `langfuse.generation()` span that records the model name, the full input messages, the output, token counts, and latency. We also accept arbitrary `metadata` tags so callers can annotate each call with pipeline stage, document ID, user session, and so on.

In [ ]:
class ObservableLLMClient(LLMClient):
    """LLMClient that records every generation to Langfuse."""

    def __init__(
        self,
        model: str = "gpt-4o-mini",
        temperature: float = 0.0,
        langfuse_client: Langfuse = None,
        trace_id: str | None = None,  # parent trace to attach generations to
    ):
        super().__init__(model=model, temperature=temperature)
        self._lf = langfuse_client or Langfuse()  # <1>
        self._trace_id = trace_id

    def complete(
        self,
        messages: list[dict],
        *,
        response_format=None,
        metadata: dict | None = None,
        generation_name: str = "llm-call",
    ):
        t0 = time.perf_counter()
        result = super().complete(messages, response_format=response_format)
        latency_ms = (time.perf_counter() - t0) * 1000

        output_str = result.model_dump_json() if hasattr(result, "model_dump_json") else str(result)

        gen = self._lf.generation(  # <2>
            name=generation_name,
            model=self.model,
            input=messages,
            output=output_str,
            usage={
                "input": self._in,
                "output": self._out,
                "unit": "TOKENS",
            },
            metadata={
                "latency_ms": round(latency_ms, 1),
                "cost_usd": self.total_cost,
                **(metadata or {}),
            },
            trace_id=self._trace_id,
        )
        gen.end()  # <3>
        return result

1. If no `langfuse_client` is passed, we create one from environment variables — this makes the client work out of the box with `.env` configuration.
2. `langfuse.generation()` opens a generation span. It accepts `model`, `input`, `output`, `usage`, and arbitrary `metadata`. The `trace_id` parameter attaches this generation as a child of an existing trace.
3. Calling `.end()` finalizes the span and flushes it to Langfuse. Without `.end()`, the span is left open and will not appear in the dashboard.

Sending a test generation to Langfuse:

In [ ]:
obs_llm = ObservableLLMClient(
    model="gpt-4o-mini",
    langfuse_client=langfuse,
)

answer = obs_llm.complete(
    messages=[
        {"role": "system", "content": "You are a financial analyst."},
        {"role": "user", "content": "What is the CET1 capital ratio regulatory minimum?"},
    ],
    generation_name="cet1-query",
    metadata={"pipeline_stage": "demo", "document_id": "10k-2024"},
)
langfuse.flush()
print(f"Answer: {answer}")
print(f"Total cost: ${obs_llm.total_cost:.6f}")
print("Check http://localhost:3000 for the trace.")

## Tracing a RAG Pipeline

A real RAG pipeline has multiple stages — retrieval, prompt construction, generation — and we want each stage to appear as a child span under a single parent trace. This lets us see not just what the LLM did, but which component was the bottleneck: if the trace shows retrieval taking 800ms and generation taking 200ms, ChromaDB is the bottleneck, not the model API.

In [ ]:
# Minimal vector store reused from notebook 03
import openai as _oai

def embed(texts: list[str], model: str = "text-embedding-3-small") -> np.ndarray:
    client = _oai.OpenAI()
    resp = client.embeddings.create(input=texts, model=model)
    return np.array([item.embedding for item in resp.data], dtype=np.float32)


CORPUS_TEXTS = [
    "Our Common Equity Tier 1 (CET1) capital ratio was 14.8% at year-end, well above the regulatory minimum of 4.5% and our internal target of 13%.",
    "We maintain a liquidity coverage ratio (LCR) of 128%, exceeding the regulatory requirement of 100%. Our high-quality liquid assets totaled $280 billion at year-end.",
    "Net revenues for the fiscal year were $47.4 billion, an increase of 8% compared to the prior year.",
    "Investment banking revenues decreased 23% to $6.1 billion, reflecting lower advisory fees.",
    "Return on equity for the year was 12.4%, compared to 15.1% in the prior year.",
]


def traced_rag_query(
    question: str,
    corpus: list[str],
    obs_llm: ObservableLLMClient,
    k: int = 2,
) -> dict:
    """RAG query with Langfuse parent trace and child spans per stage."""
    trace = langfuse.trace(  # <1>
        name="rag-query",
        input={"question": question},
        metadata={"k": k, "corpus_size": len(corpus)},
    )
    obs_llm._trace_id = trace.id

    # Stage 1: retrieval
    t0 = time.perf_counter()
    retrieval_span = trace.span(name="retrieval", input={"query": question})  # <2>
    corpus_vecs = embed(corpus)
    query_vec = embed([question])[0]
    norms = np.linalg.norm(corpus_vecs, axis=1, keepdims=True) + 1e-9
    scores = (corpus_vecs / norms) @ (query_vec / (np.linalg.norm(query_vec) + 1e-9))
    top_k_idx = np.argsort(scores)[::-1][:k]
    retrieved = [corpus[i] for i in top_k_idx]
    retrieval_latency = (time.perf_counter() - t0) * 1000
    retrieval_span.end(output={"chunks": retrieved, "latency_ms": round(retrieval_latency, 1)})

    # Stage 2: generation
    context = "\n\n".join(f"[{i+1}] {c}" for i, c in enumerate(retrieved))
    messages = [
        {"role": "system", "content": "Answer using ONLY the provided context. Cite with [N]."},
        {"role": "user", "content": f"Context:\n{context}\n\nQuestion: {question}"},
    ]
    answer = obs_llm.complete(  # <3>
        messages,
        generation_name="rag-generation",
        metadata={"n_chunks": k, "pipeline_stage": "generation"},
    )

    trace.update(output={"answer": answer})
    langfuse.flush()

    return {
        "answer": answer,
        "retrieved": retrieved,
        "trace_id": trace.id,
        "retrieval_latency_ms": round(retrieval_latency, 1),
        "total_cost_usd": obs_llm.total_cost,
    }

1. `langfuse.trace()` opens a parent trace. All child spans created with `trace.span()` or generations tagged with `trace_id=trace.id` appear nested under this parent in the dashboard.
2. `trace.span()` creates a child span for the retrieval stage. Calling `.end()` on it records the duration and output — so we can compare retrieval latency vs. generation latency in the dashboard.
3. The generation is automatically attached to the parent trace via `obs_llm._trace_id = trace.id`, so it appears as a sibling of the retrieval span.

Running a traced RAG query:

In [ ]:
result = traced_rag_query(
    question="What is the CET1 ratio and how does it compare to the minimum?",
    corpus=CORPUS_TEXTS,
    obs_llm=obs_llm,
    k=2,
)
print(f"Answer: {result['answer']}")
print(f"Retrieved chunks: {len(result['retrieved'])}")
print(f"Retrieval latency: {result['retrieval_latency_ms']:.0f} ms")
print(f"Total cost: ${result['total_cost_usd']:.6f}")
print(f"Trace ID: {result['trace_id']}")

## Prompt Versioning

Langfuse stores prompt templates with version numbers. We register two versions of the RAG system prompt, then demonstrate fetching the active version at runtime. When a regression is detected, we roll back by setting the active version back to v1 via the Langfuse API — no code deploy required.

:::{.callout-tip}
Store the active prompt version name in every trace's metadata. This is the single most valuable piece of information for incident post-mortems: "the regression started when we deployed prompt v4 on Tuesday afternoon."

:::

In [ ]:
# Register prompt versions in Langfuse
try:
    langfuse.create_prompt(
        name="rag-system-prompt",
        prompt=(
            "You are a financial analyst assistant. "
            "Answer using ONLY the provided context. "
            "Cite sources using [N] notation. "
            "If the context is insufficient, say so explicitly."
        ),
        labels=["production"],  # <1>
    )
    print("Registered rag-system-prompt v1")
except Exception:
    print("Prompt already registered (or Langfuse not reachable — skipping registration)")

try:
    langfuse.create_prompt(
        name="rag-system-prompt",
        prompt=(
            "You are a senior financial analyst assistant with expertise in SEC filings. "
            "Answer using ONLY the provided context. "
            "Cite sources using [N] notation and include specific figures. "
            "If the context is insufficient, list what additional information is needed."
        ),
        labels=["staging"],
    )
    print("Registered rag-system-prompt v2")
except Exception:
    print("v2 registration skipped")

1. The `labels` parameter in Langfuse controls which version is "live". By labelling v1 as `production`, we can fetch the production prompt at runtime without hard-coding a version number. To roll back, change the label from v2 back to v1 in the Langfuse UI — the next request automatically gets v1.

Fetching the active prompt version at runtime:

In [ ]:
def get_system_prompt(name: str = "rag-system-prompt", label: str = "production") -> str:
    """Fetch the active prompt version from Langfuse, with a hardcoded fallback."""
    FALLBACK = (
        "You are a financial analyst assistant. "
        "Answer using ONLY the provided context. Cite with [N]."
    )
    try:
        prompt_obj = langfuse.get_prompt(name, label=label)
        version = prompt_obj.version
        text = prompt_obj.prompt
        print(f"Fetched prompt '{name}' version={version} label={label}")
        return text
    except Exception as e:
        print(f"Could not fetch prompt from Langfuse ({e}). Using fallback.")
        return FALLBACK


active_prompt = get_system_prompt()
print(f"Active prompt: {active_prompt[:80]}...")

## Cost and Latency Dashboards

We simulate 20 traces with varying query complexity (more retrieved chunks = higher cost + latency) and compute percentile statistics. This is the data that populates a cost/latency SLO dashboard.

In [ ]:
import random

# Simulate trace data: vary complexity by number of context chunks
rng = random.Random(42)

simulated_traces = []
for i in range(20):
    n_chunks = rng.choice([1, 2, 3, 5, 8])  # complexity proxy
    base_latency = 400 + n_chunks * 120  # ms — more context = slower
    latency_ms = base_latency + rng.gauss(0, 80)
    input_tokens = 200 + n_chunks * 150
    output_tokens = rng.randint(80, 200)
    cost = (input_tokens * 0.15 + output_tokens * 0.60) / 1_000_000

    simulated_traces.append({
        "trace_id": f"sim-{i:03d}",
        "n_chunks": n_chunks,
        "latency_ms": max(latency_ms, 100),
        "input_tokens": input_tokens,
        "output_tokens": output_tokens,
        "cost_usd": cost,
    })

latencies = [t["latency_ms"] for t in simulated_traces]
costs = [t["cost_usd"] for t in simulated_traces]

print(f"{'Metric':<20} {'p50':>8} {'p95':>8} {'p99':>8}")
print("-" * 44)
print(f"{'Latency (ms)':<20} {np.percentile(latencies, 50):>8.0f} {np.percentile(latencies, 95):>8.0f} {np.percentile(latencies, 99):>8.0f}")
print(f"{'Cost (USD)':<20} {np.percentile(costs, 50):>8.5f} {np.percentile(costs, 95):>8.5f} {np.percentile(costs, 99):>8.5f}")

# SLO check
SLO_LATENCY_P95_MS = 3000
SLO_COST_PER_QUERY = 0.01
p95_lat = np.percentile(latencies, 95)
p95_cost = np.percentile(costs, 95)
print(f"\nSLO p95 latency < {SLO_LATENCY_P95_MS}ms: {'PASS' if p95_lat < SLO_LATENCY_P95_MS else 'FAIL'} ({p95_lat:.0f}ms)")
print(f"SLO cost/query < ${SLO_COST_PER_QUERY}: {'PASS' if p95_cost < SLO_COST_PER_QUERY else 'FAIL'} (${p95_cost:.5f})")

The simulation confirms the expected pattern: more retrieved context chunks increase both latency (more tokens to transmit and process) and cost (more input tokens billed). The SLO thresholds — p95 latency under 3 seconds, cost per query under $0.01 — are easily satisfied for the gpt-4o-mini model. We would revisit these thresholds if we switched to gpt-4o (10× more expensive) or significantly increased the context size.

:::{.callout-caution}
The simulated data above is synthetic — in production you would query the Langfuse SDK's `get_generations()` endpoint to pull real trace data. Always base SLO reviews on real production traces, not simulations.

:::

## Exercises

1. **Add a `session_id` tag.** Modify `traced_rag_query` to accept a `session_id: str` parameter and pass it as metadata on the parent trace. Run three queries with the same `session_id` and verify they appear grouped in the Langfuse dashboard under that session.

2. **Implement a cost alert function.** Write `check_cost_slo(traces: list[dict], threshold_usd: float = 0.01) -> None` that computes the p95 cost from a list of simulated trace dicts and raises a `RuntimeWarning` if the p95 cost exceeds the threshold. Test it with a `threshold_usd=0.0001` to trigger the warning.

3. **Attach a quality score to traces.** Import `GEvalScorer` from [notebook 04](/courses/llm-eng/04-eval-concepts.html) and modify `traced_rag_query` to call `scorer.score_one(question, answer, context)` after the generation step. Append the score as `metadata["quality_score"]` to the generation span so it appears alongside cost and latency in every trace.

---

$\blacksquare$